# 04 — Covariance shrinkage

Notebooks 01–03 held the covariance estimator fixed at the 252-day sample
matrix. At $T/N = 252/13 \approx 19$ that sample $\Sigma$ is the textbook
"estimation risk" regime — this notebook asks one question: **does
covariance shrinkage change allocations or performance in this universe, at
this sample size?**

Two shrinkage estimators, both `sklearn.covariance` fits already shared
through `ml.COV_ESTIMATORS`:

- **Ledoit-Wolf** (`ledoit_wolf_cov`, **registered**) — shrinks the sample
  covariance toward a scaled-identity target, with a closed-form shrinkage
  weight that minimizes expected quadratic loss.
- **OAS** (`oas_cov`, **sensitivity**) — same target, a different
  closed-form shrinkage weight; kept as a robustness check, not promoted.

The literature suggests shrinkage tends to help risk-only allocation and
gives mixed results once the objective also carries a mean estimate
("alpha"). This repo's three optimizers map onto that split directly:

| Strategy | Uses $\hat\mu$? | Where the "alpha" comes from |
|---|---|---|
| `GMV` | No | risk-only — $\Sigma$ is the entire input |
| `MaxSharpe` | Yes | the raw 252-day sample mean, unshrunk |
| `BlackLitterman(k=0.1)` | Yes, but not from `_estimate` | already shrunk hard toward the EW prior; $\Sigma$ is still the sample/shrinkage estimator |

If that split holds here, GMV should benefit most from shrinkage, MaxSharpe should gain little or lose (§6 H2 predicts ≤ 0), and BL(0.1) — whose alpha is already shrunk toward the EW prior —
should barely move at all. §6 pre-registers that as H1/H2/a predicted null
before any backtest is run.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.covariance import LedoitWolf, OAS, empirical_covariance

import maplab as ml
import _nb04_helpers as h
from maplab import Panel, apply_style, GMV, MaxSharpe, BlackLitterman, EqualWeight, backtest, summary

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

print(f"prices: {prices.shape[0]:,} days x {prices.shape[1]} assets "
      f"({prices.index.min().date()} -> {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

## §2 Estimators

`sample_cov`, `ledoit_wolf_cov` (**registered**), `oas_cov` (**sensitivity**)
— all three live in `ml.COV_ESTIMATORS` and return an annualized covariance
DataFrame. Two identities to verify before trusting the numbers downstream,
both at the 2022-12-31 snapshot (the same snapshot date notebook 03 uses):

1. **ddof identity** — `sklearn.covariance.empirical_covariance` uses
   `ddof=0`; scaling it by `252 * n/(n-1)` must reproduce `ml.sample_cov`
   (`ddof=1`) exactly.
2. **Shrinkage target** — both sklearn estimators shrink toward a scaled
   identity $mI$, $m = \mathrm{tr}(S)/N$: $\Sigma_{LW}/252$ must equal
   $(1-\delta)S + \delta m I$ for sklearn's own $\delta$ (`.shrinkage_`).

In [ ]:
asof_split = pd.Timestamp("2022-12-31")
rets_split = panel.slice(asof_split, "returns", ml.COV_LOOKBACK)[ml.UNIVERSE]
n_split = len(rets_split)
assert n_split == ml.COV_LOOKBACK, f"expected {ml.COV_LOOKBACK} rows, got {n_split}"

S = empirical_covariance(rets_split.to_numpy())
S_ddof = S * ml.TRADING_DAYS * n_split / (n_split - 1)
S_sample = ml.sample_cov(rets_split).to_numpy()
print(f"ddof identity: max|empirical_cov*252*n/(n-1) - sample_cov| = "
      f"{np.max(np.abs(S_ddof - S_sample)):.3e}")

m = np.trace(S) / len(ml.UNIVERSE)

lw_fit = LedoitWolf().fit(rets_split.to_numpy())
delta_lw_split = lw_fit.shrinkage_
target_lw = (1 - delta_lw_split) * S + delta_lw_split * m * np.eye(len(ml.UNIVERSE))
lhs_lw = ml.ledoit_wolf_cov(rets_split).to_numpy() / ml.TRADING_DAYS
print(f"LW target check:  max|Sigma_LW/252 - ((1-d)S + d*m*I)| = "
      f"{np.max(np.abs(lhs_lw - target_lw)):.3e}   (delta_LW={delta_lw_split:.4f})")

oas_fit = OAS().fit(rets_split.to_numpy())
delta_oas_split = oas_fit.shrinkage_
target_oas = (1 - delta_oas_split) * S + delta_oas_split * m * np.eye(len(ml.UNIVERSE))
lhs_oas = ml.oas_cov(rets_split).to_numpy() / ml.TRADING_DAYS
print(f"OAS target check: max|Sigma_OAS/252 - ((1-d)S + d*m*I)| = "
      f"{np.max(np.abs(lhs_oas - target_oas)):.3e}   (delta_OAS={delta_oas_split:.4f})")

## §3 Shrinkage dose through time

At every rebalance label `ml.backtest` scores (the common scoring schedule,
starting at the first eligible rebalance after warm-up): the 252-day
log-return slice strictly before the label, sklearn's LW/OAS shrinkage
weights, and the condition number of $\Sigma_{sample}$ vs $\Sigma_{LW}$.

In [ ]:
rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"
print(f"rebalance labels: n={len(rdates)}  first={rdates[0].date()}  last={rdates[-1].date()}")

dose = h.dose_table(panel, rdates)
assert len(dose) == 208
dose.describe(percentiles=[.25, .5, .75]).loc[["min", "25%", "50%", "75%", "max"]]

In [ ]:
fig = h.dose_figure(dose, split_ts)
plt.show()

## §4 Where shrinkage acts

At the three snapshot dates: $\Delta = \Sigma_{LW} - \Sigma_{sample}$
(annualized), `diag_share` $= \lVert\mathrm{diag}(\Delta)\rVert_F^2 /
\lVert\Delta\rVert_F^2$ — the share of the shrinkage move sitting on the
diagonal (vols) vs off-diagonal (correlations) — and the annualized vol
table, sample vs LW.

In [ ]:
DATES = [pd.Timestamp(d) for d in ["2014-12-31", "2017-12-31", "2022-12-31"]]

diag_share_table, vol_tables = h.diag_share_tables(panel, DATES)
diag_share_table.round(4)

In [ ]:
for asof_d, vt in vol_tables.items():
    print(f"\n{asof_d}  annualized vols (%), sample vs LW:")
    print(vt.round(2))

### `lw_corr` — snapshot-only correlation variant

Local variant that applies Ledoit-Wolf to the **correlation** structure only
and leaves sample vols untouched. Defined here for this one comparison and
never backtested — checking whether shrinking correlations alone moves
allocations differently from full LW (which also shrinks vols toward $m$).

In [ ]:
lw_corr_table = h.lw_corr_table(panel, DATES)
lw_corr_table.round(3)

## §5 Design checks

### (a) BL(k=0) = EW under every estimator

In [ ]:
bl_k0_rows = []
for asof in DATES:
    for est_name, est_fn in [("sample", ml.sample_cov), ("ledoit_wolf", ml.ledoit_wolf_cov), ("oas", ml.oas_cov)]:
        w = BlackLitterman(cov_estimator=est_fn, k=0.0)(panel, asof)
        max_dev = float((w - 1.0 / 13).abs().max())
        bl_k0_rows.append({"asof": asof.date(), "estimator": est_name, "max_abs_w_minus_1_13": max_dev})

bl_k0_table = pd.DataFrame(bl_k0_rows).set_index(["asof", "estimator"])
bl_k0_table

### (b) Itô term

`GMV.predict_weights` uses only `Sigma` from `_estimate` (the `mu` return
value is discarded entirely). `BlackLitterman.posterior` also calls
`self._estimate(panel, asof)` but only keeps `Sigma`; `mu_bl` is built
separately from the EW prior and momentum views, never from `_estimate`'s
`mu`. So the Itô correction below (adding $\tfrac12\sigma^2$ using the
estimator's *own* variance instead of the sample variance) can only matter
for `MaxSharpe`, the one strategy that optimizes directly on `_estimate`'s
`mu`.

In [ ]:
ito_table = h.ito_table(panel, DATES)
ito_table.round(4)

**Pre-set rule:** apply the Itô fix to the shared `_estimate` code only if
`max|dmu| > 10 bp/yr` OR `halfL1(dw) > 1%` on any date. Default expectation
is no fix — the verdict below is read directly off the numbers in the table
above.

In [ ]:
ito_fix_needed = bool((ito_table["max_abs_dmu_bp"] > 10).any() or (ito_table["halfL1_dw_%"] > 1.0).any())
print(f"max|dmu| observed = {ito_table['max_abs_dmu_bp'].max():.4f} bp/yr")
print(f"max halfL1(dw) observed = {ito_table['halfL1_dw_%'].max():.4f}%")
print(f"Verdict: {'APPLY Ito fix' if ito_fix_needed else 'NO Ito fix -- MaxSharpe._estimate stays as-is.'}")

### (c) Weights: sample vs LW vs OAS

GMV, MaxSharpe, BL(0.1) at the 3 dates, fresh strategy instances per
estimator.

In [ ]:
weight_table, weight_frames = h.weight_tables(panel, DATES)
weight_table.round(3)

In [ ]:
for (asof_d, strat_name), wf in weight_frames.items():
    print(f"\n{asof_d}  {strat_name}  (weights, %)")
    print(wf.round(2))

### (d) Concentration at 2022-12-31

Concentration that survives shrinkage is structural, not estimation noise:
if GMV and MaxSharpe stay pinned near the same corner under sample, LW,
*and* OAS, that corner is a feature of the 2022 risk-off correlation
structure, not sampling error the shrinkage estimators are correcting.

In [ ]:
concentration_2022 = weight_table.loc[
    (pd.Timestamp("2022-12-31").date(), ["GMV", "MaxSharpe"], slice(None)),
    ["effN", "max_w_%", "argmax"],
]
concentration_2022

In [ ]:
ml.inference.render_registration(ml.inference.load_registration("nb04"))

## §7 Backtests

Fresh instances: GMV, MaxSharpe, BL(k=0.1) × {sample, ledoit_wolf, oas} +
EqualWeight — 10 runs total, same cost/rebalance/warm-up harness as
notebook 03.

In [ ]:
strategies = {
    "GMV(S)": GMV(cov_estimator=ml.sample_cov),
    "GMV(LW)": GMV(cov_estimator=ml.ledoit_wolf_cov),
    "GMV(OAS)": GMV(cov_estimator=ml.oas_cov),
    "MS(S)": MaxSharpe(cov_estimator=ml.sample_cov),
    "MS(LW)": MaxSharpe(cov_estimator=ml.ledoit_wolf_cov),
    "MS(OAS)": MaxSharpe(cov_estimator=ml.oas_cov),
    "BL(S)": BlackLitterman(cov_estimator=ml.sample_cov, k=0.1),
    "BL(LW)": BlackLitterman(cov_estimator=ml.ledoit_wolf_cov, k=0.1),
    "BL(OAS)": BlackLitterman(cov_estimator=ml.oas_cov, k=0.1),
    "EW": EqualWeight(),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

print(f"{len(results)} runs completed.")

### Reproduction gate

The sample-estimator runs and EW must match notebooks 01/03 in full/train/test
Sharpe to 2 dp: GMV 0.62/0.59/0.76, MaxSharpe 0.72/0.60/1.13,
BL(0.1) 0.64/0.62/0.70, EW 0.74/0.70/0.91. If this doesn't hold the harness
or estimator wiring changed something and everything downstream is suspect —
so this stops the notebook (`assert`) rather than just printing a warning.

In [ ]:
rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel]
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]
summary_table["cost_drag"] = summary_table["ann_turnover"] * ml.COST_BPS / 1e4
summary_table.round(4)

In [ ]:
ml.inference.check_reproduction(summary_table, {"GMV(S)": "GMV_sample", "MS(S)": "MaxSharpe_sample", "BL(S)": "BL_k0.1_sample", "EW": "EW"})

print("REPRODUCTION GATE: PASSED -- sample-estimator runs and EW match notebooks 01/03 to 2 dp.")

In [ ]:
summary_table_display = summary_table.reset_index()
summary_table_display["group"] = summary_table_display["strategy"].map(h.run_group)
summary_table_display = summary_table_display.set_index(["group", "strategy", "window"]).sort_index()
summary_table_display.round(4)

In [ ]:
bookkeeping_rows = []
for name, r in results.items():
    strat = r["strat"]
    row = {"strategy": name}
    for attr in ["fallback_dates", "retry_dates", "concentration_dates"]:
        if hasattr(strat, attr):
            row[attr] = len(getattr(strat, attr))
    bookkeeping_rows.append(row)
pd.DataFrame(bookkeeping_rows).set_index("strategy")

In [ ]:
print("Sharpe standard error ~ sqrt((1+SR^2/2)/T), T in years (same convention as notebooks 01-03):")
for name, r in results.items():
    net_full = r["net"]
    net_test = net_full.loc[net_full.index > split_ts]
    sr_full = float(summary_table.loc[(name, "full"), "sharpe"])
    sr_test = float(summary_table.loc[(name, "test (> split)"), "sharpe"])
    T_full = len(net_full) / ml.TRADING_DAYS
    T_test = len(net_test) / ml.TRADING_DAYS
    se_full = ml.inference.sharpe_se(sr_full, T_full)
    se_test = ml.inference.sharpe_se(sr_test, T_test)
    print(f"  {name:10s} full: SR={sr_full:.2f} SE={se_full:.2f}   |   test: SR={sr_test:.2f} SE={se_test:.2f}")

## §8 Paired differences

For each strategy family, LW − sample (registered) and OAS − sample
(sensitivity): annualized mean difference, tracking error, $t$-stat,
full/train/test, plus the cost-drag difference and the gross-of-costs mean
difference (same pattern as notebook 03's active-return-vs-EW block, here
applied to same-family, different-estimator pairs instead of strategy-vs-EW).

In [ ]:
pairs = [
    ("GMV(LW)", "GMV(S)", "registered"), ("GMV(OAS)", "GMV(S)", "sensitivity"),
    ("MS(LW)", "MS(S)", "registered"), ("MS(OAS)", "MS(S)", "sensitivity"),
    ("BL(LW)", "BL(S)", "registered"), ("BL(OAS)", "BL(S)", "sensitivity"),
]

paired_rows = []
paired_series = {}
for est_name, base_name, kind in pairs:
    net_est = results[est_name]["net"]
    net_base = results[base_name]["net"]
    idx = net_est.index.intersection(net_base.index)
    diff = net_est.loc[idx] - net_base.loc[idx]
    paired_series[(est_name, base_name)] = diff
    win_defs = {
        "full": diff.index >= diff.index.min(),
        "train (<= split)": diff.index <= split_ts,
        "test (> split)": diff.index > split_ts,
    }
    for window_name, sel in win_defs.items():
        d = diff.loc[sel]
        ann_mean = float(d.mean() * ml.TRADING_DAYS)
        te = float(d.std(ddof=1) * np.sqrt(ml.TRADING_DAYS))
        t_stat = float(d.mean() / (d.std(ddof=1) / np.sqrt(len(d))))
        cost_drag_est = float(summary_table.loc[(est_name, window_name), "cost_drag"])
        cost_drag_base = float(summary_table.loc[(base_name, window_name), "cost_drag"])
        cost_drag_diff = cost_drag_est - cost_drag_base
        gross_diff = ann_mean + cost_drag_diff
        paired_rows.append({
            "pair": f"{est_name} - {base_name}", "kind": kind, "window": window_name,
            "ann_mean_diff": ann_mean, "TE": te, "t": t_stat,
            "cost_drag_diff": cost_drag_diff, "gross_of_costs_mean_diff": gross_diff,
        })

paired_table = pd.DataFrame(paired_rows).set_index(["pair", "kind", "window"])
paired_table.round(4)

In [ ]:
fig = h.cumulative_lw_figure(paired_series, split_ts)
plt.show()

In [ ]:
fig = h.halfl1_lw_figure(results, split_ts)
plt.show()

## §9 $\beta$ decomposition

Plain OLS (no HAC — same caveat as notebook 03), realized daily simple
excess return on the market's simple excess return, for each sample and LW
run. For each LW − sample pair, decompose the diff into a $\beta$-shift
term ($\Delta\beta \times$ mean market excess) and an $\alpha$-shift term
($\Delta\alpha$); the identity check below is 0 by OLS construction (mean
residual is exactly 0 for an OLS fit with an intercept), not a residual to
be interpreted.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_table = ml.inference.capm_table(results, ["GMV(S)", "GMV(LW)", "MS(S)", "MS(LW)", "BL(S)", "BL(LW)"],
                                     MKT_simple, rf_daily)
capm_table.round(4)

In [ ]:
for est_name, base_name, label in [("GMV(LW)", "GMV(S)", "GMV"), ("MS(LW)", "MS(S)", "MaxSharpe"), ("BL(LW)", "BL(S)", "BL(0.1)")]:
    alpha_base = float(capm_table.loc[base_name, "alpha_ann"])
    beta_base = float(capm_table.loc[base_name, "beta"])
    alpha_est = float(capm_table.loc[est_name, "alpha_ann"])
    beta_est = float(capm_table.loc[est_name, "beta"])

    net_est = results[est_name]["net"]
    idx = net_est.index.intersection(rf_daily.index).intersection(MKT_simple.index)
    mkt_ann_mean = float(MKT_simple.loc[idx].mean() * ml.TRADING_DAYS)

    d_alpha = alpha_est - alpha_base
    d_beta = beta_est - beta_base
    beta_part = d_beta * mkt_ann_mean
    decomposed = d_alpha + beta_part

    realized_diff_full = float(paired_table.loc[(f"{est_name} - {base_name}", slice(None), "full"), "ann_mean_diff"].iloc[0])
    identity_check = realized_diff_full - decomposed

    print(f"=== {label}: LW - sample (full window) ===")
    print(f"  d_alpha (ann)         = {d_alpha:.4%}")
    print(f"  d_beta                = {d_beta:.4f}")
    print(f"  ann mean MKT excess   = {mkt_ann_mean:.4%}")
    print(f"  beta_part = d_beta * mkt_ann_mean = {beta_part:.4%}")
    print(f"  d_alpha + beta_part   = {decomposed:.4%}")
    print(f"  realized ann_mean_diff (Sec 8, full) = {realized_diff_full:.4%}")
    print(f"  identity check (0 by OLS construction) = {identity_check:.4%}")
    print()

## §10 TAKEAWAY

- **Small δ, large conditioning effect.** $\delta_{LW}$ median 0.0368 (range
  0.0198–0.1336, max at 2020-03-31); median condition number falls from
  512.7 to 119.9.
- **Shrinkage acts mostly on correlations.** LW diag share 27.3%/33.3%/12.9%
  at 2014-12-31/2017-12-31/2022-12-31 — 65–87% of the shrinkage move is
  off-diagonal at all three dates; `lw_corr` stays snapshot-only.
- **Allocation effect:** median ½L1 LW vs sample: GMV 5.10%, MaxSharpe 3.25%,
  BL 1.73% — GMV moves the most, BL the least, consistent with how much
  "alpha" each objective is carrying.
- **H1 (GMV): passes |t| ≥ 2 on the full window only — not robust.** LW − S
  +0.25%/yr full (t = 2.34); train is a subset of full, not independent;
  test +0.17%/yr (t = 0.94); OAS +0.06%/yr (t = 1.14). Plain-OLS t, no HAC,
  3 pairs tested. Not a cost effect: cost-drag difference −0.02%/yr
  (turnover 70.7%/yr → 54.9%/yr), gross-of-costs +0.24%/yr. Not a risk
  effect either: realized vol 3.17% → 3.19%. Δα 0.16% vs Δβ part 0.10%.
- **H2 (MaxSharpe): not supported — sign opposite to the prediction, not
  significant.** LW − S +0.40%/yr full (t = 1.61), test +0.46%/yr
  (t = 0.67); Δα 0.28% vs Δβ part 0.12%. Sharpe full 0.72 → 0.73, test
  1.13 → 1.05: higher return with higher vol, so risk-adjusted performance
  is unchanged.
- **BL(0.1):** LW − S 0.07%/yr (t = 1.00) → null as predicted.
- **OAS sensitivity:** agrees with LW on sign for GMV (t = 1.14) and
  MaxSharpe (t = 1.53), both sub-|t|=2; BL(OAS) − BL(S) is ≈0.00%/yr
  (t = −0.05) vs BL(LW)'s 0.07% — both are economically indistinguishable
  from zero, so the "sign flip" there is noise around a null, not a
  disagreement.
- **Corners survive shrinkage.** 2022-12-31 GMV max 55.35% UUP (effN 2.77)
  vs sample 58.12%; MaxSharpe max 74.09% UUP (effN 1.62) vs sample 74.41%.
  The concentration is not sample-covariance noise; its cause is not tested
  here.
- **Itô term:** max 7.16 bp/yr, 0.26% ½L1 → no fix to shared code.